# Consultas Analíticas SQL

## 1. Objetivo

Desenvolver consultas SQL sobre as tabelas da camada Gold, explorando indicadores financeiros e informações de negócio relacionadas às transações PIX.

As consultas utilizam as tabelas Delta persistidas no catálogo Databricks e demonstram operações de agregação, ordenação, junção lógica e funções analíticas.

## 2. Fontes de dados

As consultas deste notebook utilizam as seguintes tabelas:

- `workspace.gold.gold_transacoes_pix`: visão detalhada das transações PIX, enriquecida com informações de contas e clientes.
- `workspace.gold.gold_resumo_cliente`: indicadores financeiros consolidados por cliente.

A camada Gold é utilizada como camada de consumo analítico, evitando repetir neste notebook as transformações realizadas nas camadas Bronze e Silver.

## 3. Top 10 clientes por volume financeiro

**Objetivo:** identificar os dez clientes com maior valor total transacionado.
A consulta utiliza a tabela de resumo por cliente e ordena os resultados pelo total transacionado em ordem decrescente.

In [0]:
%sql
SELECT
    id_cliente,
    nome_cliente,
    cidade,
    estado,
    total_transacionado,
    quantidade_transacoes,
    ticket_medio
FROM workspace.gold.gold_resumo_cliente
ORDER BY total_transacionado DESC
LIMIT 10;

## 4. Movimentação financeira por dia

**Objetivo:** acompanhar a quantidade de transações e o volume financeiro movimentado em cada dia.

Cada linha representa uma data com movimentação registrada na tabela de transações.

In [0]:
%sql
SELECT
    data_transacao,
    COUNT(*) AS quantidade_transacoes,
    SUM(valor) AS valor_total_transacionado
FROM workspace.gold.gold_transacoes_pix
GROUP BY data_transacao
ORDER BY data_transacao;

## 5. Ticket médio por estado

**Objetivo:** comparar o ticket médio das transações PIX entre os estados.

Além do ticket médio, a consulta apresenta a quantidade de transações e o volume financeiro total por estado.

In [0]:
%sql
SELECT
    estado,
    COUNT(*) AS quantidade_transacoes,
    ROUND(AVG(valor), 2) AS ticket_medio,
    ROUND(SUM(valor), 2) AS valor_total
FROM workspace.gold.gold_transacoes_pix
GROUP BY estado
ORDER BY ticket_medio DESC;

## 6. Clientes sem transações

**Objetivo:** identificar clientes que não realizaram nenhuma transação PIX.

A tabela `gold_resumo_cliente` preserva todos os clientes da Silver. Para clientes sem movimentação, a quantidade de transações é preenchida com zero durante a construção da Gold.

**Observação sobre os dados utilizados:** a execução desta consulta não retornou linhas, pois todos os 24 clientes presentes na amostra possuem pelo menos uma transação. O resultado vazio é válido e não indica falha na consulta.

In [0]:
%sql
SELECT
    id_cliente,
    nome_cliente,
    cidade,
    estado
FROM workspace.gold.gold_resumo_cliente
WHERE quantidade_transacoes = 0
ORDER BY nome_cliente;

## 7. Ranking de cidades por volume financeiro

**Objetivo:** classificar as cidades de acordo com o volume financeiro movimentado em transações PIX.

Primeiro, a CTE consolida a quantidade de transações e o volume financeiro por cidade e estado. Em seguida, `DENSE_RANK()` atribui posições em ordem decrescente de volume. Cidades com o mesmo volume recebem a mesma posição, sem lacunas na sequência do ranking.

In [0]:
%sql
WITH volume_cidades AS (
    SELECT
        cidade,
        estado,
        COUNT(*) AS quantidade_transacoes,
        SUM(valor) AS volume_financeiro
    FROM workspace.gold.gold_transacoes_pix
    GROUP BY cidade, estado
)

SELECT
    cidade,
    estado,
    quantidade_transacoes,
    ROUND(volume_financeiro, 2) AS volume_financeiro,
    DENSE_RANK() OVER (
        ORDER BY volume_financeiro DESC
    ) AS ranking
FROM volume_cidades
ORDER BY ranking;